In [1]:
import pandas as pd
import numpy as np
import re
from typing import Tuple, List, Optional, Dict
from scipy.stats import spearmanr

import warnings
warnings.filterwarnings("ignore")

In [2]:
def load_and_process_data(in_vitro: str = "data/in_vitro_modeling.csv", marmoset: str = "data/marmoset_wide_clustered_classif.csv") -> Tuple[pd.DataFrame, pd.DataFrame]:
    """Load and process data for in vitro and marmoset datasets."""
    try: 
        in_vitro = pd.read_csv(in_vitro)
        marmoset = pd.read_csv(marmoset)
        
        print(f"In vitro data shape: {in_vitro.shape}")
        print(f"Marmoset data shape: {marmoset.shape}")
        
        return in_vitro, marmoset

    except Exception as e:
        print(f"Error loading data: {e}")
        return None, None

In [3]:
def identify_pathology_features(marmoset_df: pd.DataFrame) -> Tuple[List[str], List[str], List[str]]:
    """
    Identify pathology features in marmoset dataset.
    Returns a list of:
    - All common features between TP2 and TP6
    - TP2 feature columns
    - TP6 feature columns
    """
    
    all_cols = marmoset_df.columns.tolist()
    
    # feature selection
    tp2_cols = [c for c in all_cols if c.startswith('TP2_') and not any(x in c for x in ['LesionType', 'FinalLesionType', 'Consolidate', 'Contour'])]
    tp6_cols = [c for c in all_cols if c.startswith('TP6_') and not any(x in c for x in ['LesionType', 'FinalLesionType', 'Consolidate', 'Contour'])]

    # base feature names
    tp2_base = [c.replace('TP2_', '') for c in tp2_cols]
    tp6_base = [c.replace('TP6_', '') for c in tp6_cols]

    # common features
    common_features = list(set(tp2_base) & set(tp6_base))
    
    print(f"Found {len(common_features)} common features between TP2 and TP6.")
    
    return common_features, tp2_cols, tp6_cols

In [4]:
def calculate_deltas(marmoset_df: pd.DataFrame, common_features: List[str]) -> pd.DataFrame:
    """
    Calculate deltas (TP6-TP2) for pathology features.
    Deltas are calculated in such a way that negative values indicate improvement.
    """
    
    delta_df = marmoset_df.copy()
    
    for feature in common_features:
        tp6_col = f'TP6_{feature}'
        tp2_col = f'TP2_{feature}'
        delta_col = f'delta_{feature}'
        
        if tp6_col in marmoset_df.columns and tp2_col in marmoset_df.columns:
            delta_df[delta_col] = marmoset_df[tp6_col] - marmoset_df[tp2_col]
    
    return delta_df

In [5]:
def get_correlation_features(in_vitro_df: pd.DataFrame, marmoset_df: pd.DataFrame, common_features: List[str]) -> Tuple[List[str], List[str]]:
    """Get lists of features for correlation analysis."""
    
    # in vitro features
    in_vitro_features = [col for col in in_vitro_df.columns if col != 'Drug']
    
    # TP6 features
    tp6_features = [f'TP6_{feature}' for feature in common_features]
    
    # TP2 features
    tp2_features = [f'TP2_{feature}' for feature in common_features]
    
    # delta features
    delta_features = [f'delta_{feature}' for feature in common_features]
    
    # all pathology features
    all_pathology_features = tp6_features + delta_features # TP2 is start of treatment so meaningless for correlations
    
    print(f"In vitro features: {len(in_vitro_features)}")
    print(f"TP6 features: {len(tp6_features)}")
    print(f"Delta features: {len(delta_features)}")
    
    return in_vitro_features, all_pathology_features

In [6]:
def filter_in_vitro_features(features: List[str],
                            exclude_prefixes: List[str] = None,
                            exclude_substrings: List[str] = None) -> List[str]:
    exclude_prefixes = exclude_prefixes or []
    exclude_substrings = exclude_substrings or []
    out = []
    
    for f in features:
        if any(f.startswith(p) for p in exclude_prefixes):
            continue
        if any (s in f for s in exclude_substrings):
            continue
        out.append(f)
    
    return out

In [7]:
def calculate_correlations_by_severity_aggregated(
    in_vitro_df: pd.DataFrame,
    delta_df: pd.DataFrame,
    in_vitro_features: List[str],
    pathology_features: List[str],
    severity: str,
    agg: str = 'mean',
    n_boot: int = 500,
    ci: float = 0.95,
    random_state: Optional[int] = 42
) -> Dict[str, np.ndarray]:
    """
    Aggregate lesion-level pathology features to the compound level (mean/median)
    and compute Spearman correlations across compounds. Optionally perform
    block bootstrap over compounds to obtain percentile CIs and empirical p-values.

    Returns a dict of matrices keyed by: 'rho', 'p', 'n', 'ci_low', 'ci_high', 'boot_p'.
    """
    if agg not in {'mean', 'median'}:
        raise ValueError("agg must be 'mean' or 'median'")

    # filter by severity
    severity_data = delta_df[delta_df['classif'] == severity].copy()
    if len(severity_data) == 0:
        raise ValueError(f"No data found for severity: {severity}. Options are: {delta_df['classif'].unique()}.")

    # uppercase for joins
    in_vitro_df = in_vitro_df.copy()
    in_vitro_df['COMP_UP'] = in_vitro_df['Drug'].str.upper()
    severity_data = severity_data.copy()
    severity_data['COMP_UP'] = severity_data['Compound'].str.upper()

    # common compounds
    common_compounds = sorted(set(in_vitro_df['COMP_UP']) & set(severity_data['COMP_UP']))
    if len(common_compounds) < 3:
        print(f"[WARNING]: Only {len(common_compounds)} common compounds found for {severity} severity.")
        return {k: None for k in ['rho','p','n','ci_low','ci_high','boot_p']}

    print(f"Aggregated correlation for {severity} lesions across {len(common_compounds)} compounds using {agg}.")

    # aggregate pathology per compound for all requested features
    grouped = severity_data[severity_data['COMP_UP'].isin(common_compounds)].groupby('COMP_UP')[pathology_features]
    if agg == 'mean':
        path_agg = grouped.mean()
    else:
        path_agg = grouped.median()

    # prepare result matrices
    m = len(in_vitro_features)
    n = len(pathology_features)
    rho_matrix = np.full((m, n), np.nan)
    p_matrix = np.full((m, n), np.nan)
    n_matrix = np.zeros((m, n), dtype=int)
    ci_low_matrix = np.full((m, n), np.nan)
    ci_high_matrix = np.full((m, n), np.nan)
    boot_p_matrix = np.full((m, n), np.nan)

    rng = np.random.default_rng(random_state) if n_boot and n_boot > 0 else None
    alpha = 1 - ci

    # build in vitro feature frame keyed by compound
    iv_all = in_vitro_df.set_index('COMP_UP')

    for i, iv_feature in enumerate(in_vitro_features):
        iv_series_full = iv_all[iv_feature] if iv_feature in iv_all.columns else None
        if iv_series_full is None:
            continue
        for j, path_feature in enumerate(pathology_features):
            # intersect compounds with non-missing data for this pair
            path_col = path_agg[path_feature]
            both = pd.concat([iv_series_full, path_col], axis=1, join='inner').dropna()
            both = both.loc[both.index.intersection(common_compounds)]
            if len(both) < 3:
                continue
            iv_vals = both[iv_feature].to_numpy()
            path_vals = both[path_feature].to_numpy()
            n_comp = len(both)
            n_matrix[i, j] = n_comp
            try:
                rho, pval = spearmanr(iv_vals, path_vals)
            except Exception:
                rho, pval = np.nan, np.nan
            rho_matrix[i, j] = rho
            p_matrix[i, j] = pval

            # bootstrap over compounds (block bootstrap)
            if rng is not None and n_comp >= 3:
                boots = []
                idx = np.arange(n_comp)
                for _ in range(n_boot):
                    print(f"bootstrap {_} of {n_boot}")
                    sample_idx = rng.choice(idx, size=n_comp, replace=True)
                    try:
                        r_b, _ = spearmanr(iv_vals[sample_idx], path_vals[sample_idx])
                    except Exception:
                        r_b = np.nan
                    boots.append(r_b)
                boots = np.array(boots, dtype=float)
                boots = boots[~np.isnan(boots)]
                if len(boots) > 10:
                    lo = np.quantile(boots, alpha/2)
                    hi = np.quantile(boots, 1 - alpha/2)
                    ci_low_matrix[i, j] = lo
                    ci_high_matrix[i, j] = hi
                    # empirical two-sided p-value relative to |rho_obs|
                    boot_p = np.mean(np.abs(boots) >= (0 if np.isnan(rho) else abs(rho)))
                    boot_p_matrix[i, j] = boot_p

    return {
        'rho': rho_matrix,
        'p': p_matrix,
        'n': n_matrix,
        'ci_low': ci_low_matrix,
        'ci_high': ci_high_matrix,
        'boot_p': boot_p_matrix
    }

In [8]:
def save_correlation_tables_extended(
    results: Dict[str, np.ndarray],
    in_vitro_features: List[str],
    pathology_features: List[str],
    severity_name: str,
    agg: str
) -> None:
    if results['rho'] is None:
        print(f"No valid aggregated correlations calculated for {severity_name}.")
        return
    def to_df(mat):
        df = pd.DataFrame(mat, index=in_vitro_features, columns=pathology_features)
        df.index.name = 'feature'
        return df

    suffix = f"_{agg}"
    to_df(results['rho']).to_csv(f"{severity_name}_rho_values{suffix}.csv")
    to_df(results['p']).to_csv(f"{severity_name}_p_values{suffix}.csv")
    to_df(results['n']).to_csv(f"{severity_name}_n_pairs{suffix}.csv")
    to_df(results['ci_low']).to_csv(f"{severity_name}_rho_ci_low{suffix}.csv")
    to_df(results['ci_high']).to_csv(f"{severity_name}_rho_ci_high{suffix}.csv")
    to_df(results['boot_p']).to_csv(f"{severity_name}_boot_p_values{suffix}.csv")
    print(f"Saved aggregated correlation tables for {severity_name} ({agg}).")


## Correlation Directionality:
- IN VITRO FEATURES:
    - FxC: negative = synergistic (good) | positive = antagonistic (bad)
    - AUC: closer to 0 = more potent (good) | closer to 1 = less potent (bad)
    - GRinf: negative = good | positive = bad
    - Einf: closer to 1 = more potent (good) | closer to 0 = less potent (bad)

- PATHOLOGY FEATURES:
    - TP6 values: lower = better outcome (good) | higher = worse outcome (bad)
    - Delta (TP6-TP2): negative = improvement (good) | positive = worsening (bad)
    - CFU: closer to 0 = less bugs (good) | higher = more bugs (bad)

For a MATCHING correlation (good in vitro correlated with good outcome):
    - Case 1: FxC, AUC, GRinf: a positive correlation is good (both decrease together)
    - Case 2: Einf: a negative correlation is good (Einf increases, TP6 decreases)

In [12]:
IN_VITRO_DF = "data/in_vitro_combos.csv"
MARMOSET_DF = "data/marm_data_wide_clustered_classif.csv"

in_vitro_df, marmoset_df = load_and_process_data(IN_VITRO_DF, MARMOSET_DF)

common_features, tp2_cols, tp6_cols = identify_pathology_features(marmoset_df)
delta_df = calculate_deltas(marmoset_df, common_features)

in_vitro_features, pathology_features = get_correlation_features(in_vitro_df, delta_df, common_features)

in_vitro_features = filter_in_vitro_features(in_vitro_features,
                                            exclude_prefixes=["IC50", "IC90", 
                                                            "FBC50" ,"FBC90",
                                                            "LoeweFIC50", "LoeweFIC90",
                                                            "AUC25"
                                                            ])

for severity in ['cool', 'hot']:
    severity_canon = 'less_severe' if severity == 'cool' else 'severe'
    print(f"Analyzing {severity_canon} lesions...")
    
    res = calculate_correlations_by_severity_aggregated(
        in_vitro_df, delta_df, in_vitro_features, pathology_features, severity, agg="mean", n_boot=10
    )
    
    save_correlation_tables_extended(res, in_vitro_features, pathology_features, severity_canon, agg="mean")

In vitro data shape: (10, 271)
Marmoset data shape: (1193, 129)
Found 17 common features between TP2 and TP6.
In vitro features: 270
TP6 features: 17
Delta features: 17
Analyzing less_severe lesions...
Aggregated correlation for cool lesions across 10 compounds using mean.
bootstrap 0 of 10
bootstrap 1 of 10
bootstrap 2 of 10
bootstrap 3 of 10
bootstrap 4 of 10
bootstrap 5 of 10
bootstrap 6 of 10
bootstrap 7 of 10
bootstrap 8 of 10
bootstrap 9 of 10
bootstrap 0 of 10
bootstrap 1 of 10
bootstrap 2 of 10
bootstrap 3 of 10
bootstrap 4 of 10
bootstrap 5 of 10
bootstrap 6 of 10
bootstrap 7 of 10
bootstrap 8 of 10
bootstrap 9 of 10
bootstrap 0 of 10
bootstrap 1 of 10
bootstrap 2 of 10
bootstrap 3 of 10
bootstrap 4 of 10
bootstrap 5 of 10
bootstrap 6 of 10
bootstrap 7 of 10
bootstrap 8 of 10
bootstrap 9 of 10
bootstrap 0 of 10
bootstrap 1 of 10
bootstrap 2 of 10
bootstrap 3 of 10
bootstrap 4 of 10
bootstrap 5 of 10
bootstrap 6 of 10
bootstrap 7 of 10
bootstrap 8 of 10
bootstrap 9 of 10
bootst

In [13]:
# Combine mean-aggregated outputs into legacy-style files for plotting (data/in_vitro_*.csv)
# Reads the four mean CSVs and writes combined p/rho tables matching Figure3_correlations expectations.
less_p = pd.read_csv('less_severe_p_values_mean.csv')
less_r = pd.read_csv('less_severe_rho_values_mean.csv')
sev_p  = pd.read_csv('severe_p_values_mean.csv')
sev_r  = pd.read_csv('severe_rho_values_mean.csv')

# Ensure 'feature' column exists and drop empties
for df in (less_p, less_r, sev_p, sev_r):
    if 'feature' not in df.columns:
        raise ValueError('Expected a feature column in aggregated outputs')
    df.dropna(subset=['feature'], inplace=True)

# Determine pathology columns from less severe files
path_cols = [c for c in less_r.columns if c != 'feature']

# Build combined RHO table: less columns as-is, severe columns with '2' suffix
rho_less = less_r.copy()
rho_sev  = sev_r.copy()
rho_sev.rename(columns={c: f'{c}2' for c in path_cols}, inplace=True)
rho_comb = rho_less.merge(rho_sev[['feature'] + [f'{c}2' for c in path_cols]], on='feature', how='inner')

# Build combined P-VALUE table: less with '_p', severe with '_p2'
p_less = less_p.copy()
p_sev  = sev_p.copy()
p_less.rename(columns={c: f'{c}_p' for c in path_cols}, inplace=True)
p_sev.rename(columns={c: f'{c}_p2' for c in path_cols}, inplace=True)
p_comb = p_less.merge(p_sev[['feature'] + [f'{c}_p2' for c in path_cols]], on='feature', how='inner')

# Save to legacy locations
p_out   = 'data/v2_in_vitro_p_values.csv'
rho_out = 'data/v2_in_vitro_rho_values.csv'
rho_comb.to_csv(rho_out, index=False)
p_comb.to_csv(p_out, index=False)
print(f'Saved combined tables: {rho_out}, {p_out}')


Saved combined tables: data/v2_in_vitro_rho_values.csv, data/v2_in_vitro_p_values.csv
